In [2]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
import polars as pl

In [3]:
# The data for this is housed on my Mac Mini on my network. And since it is 400GB+ data 
# I cannot upload it to GitHub. So I will just provide the code and you can run it on 
# your own machine if you have the data.

DATA_FOLDER = Path("/Volumes/EXTREME_SSD/data/deribit_option")
OPTIONS_FOLDER = DATA_FOLDER / "option"
UNDERLYING_FOLDER = DATA_FOLDER / "underlying"
TRADES_FOLDER = DATA_FOLDER / "option_trades"

In [4]:
# Alternatively, we only have order book data of 10 levels from 2026/09/06.
# Before that it was just the top of the book. And since this notebook is
# focused on the Avellaneda Stoikov model, we will only use the order 
# book data from 2026/09/06 and onwards.
CUTOFF_DATE = "20260906"

# Get size of the data folders after this.
# They are saved as YYYYMMSS
def filter_sub_folders(dir_path: str):
    folders = [f for f in os.listdir(dir_path) if os.path.isdir(os.path.join(dir_path, f)) and f >= CUTOFF_DATE]
    return [os.path.join(dir_path, f) for f in folders]

opt_folders     = filter_sub_folders(OPTIONS_FOLDER)
und_folders     = filter_sub_folders(UNDERLYING_FOLDER)
opt_trds_folders = filter_sub_folders(TRADES_FOLDER)

print("Number of option folders:", len(opt_folders))
print("Number of underlying folders:", len(und_folders))
print("Number of option trades folders:", len(opt_trds_folders))

Number of option folders: 1
Number of underlying folders: 1
Number of option trades folders: 1


In [6]:
# All data is saved as parquet files. They will be aggregated into one day.parquet
# for each day. But this runs at the end of day, so we need to handle the case
# when the day is not yet finished. So we use * to get all the parquet files 
# for each day.
#
# We will also use polars for this since the the data is huge and we need to
# handle this lazily to not exhaust RAM and crash the machine.
def folders_to_parquet_globs(folders: list[str]):
    return [os.path.join(f, "*.parquet") for f in folders]

opt_lf = pl.scan_parquet(folders_to_parquet_globs(opt_folders))

opt_df = (
    opt_lf
    .filter(pl.col("bid_prices").is_not_null())
    .collect()
)
opt_df.shape

ColumnNotFoundError: unable to find column "bid_prices"; valid columns: ["datetime", "symbol", "underlying", "side", "settlement_type", "strike_price", "expiry_datetime", "last_price", "mark_price", "bid_price", "ask_price", "bid_qty", "ask_qty"]

Did you mean "bid_price"?

Resolved plan until failure:

	---> FAILED HERE RESOLVING THIS_NODE <---
Parquet SCAN [/Volumes/EXTREME_SSD/data/deribit_option/option/20260906/000000.parquet, ... 68387 other sources]
PROJECT */13 COLUMNS
ESTIMATED ROWS: 49755523